## Dataset Overview, Validation and Distribution Analysis

**Import libraries**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**Load dataset**

In [2]:
DATASET="../dataset/SmartHomeIoTNLU.csv"

df=pd.read_csv(DATASET)

df.head()

,Event_ID,Intent_Type,Command,Device,Current State,Time,Light,Temperature,Noise,Occupancy,Location,Scenario,Action,Parameters
0,a3fdd58f-6f55-493e-ba54-5c2eb0173e82,Scene_Control,Time to get up,SmartPlug_TV,On,Morning,71%,17°C,High,2,Bedroom,Wakeup_Bedroom,On,NaN
1,9bdef847-288e-4c18-b502-5a4547a9759c,Scene_Control,Shutdown study room,Mop Robot,On,Morning,82%,13°C,Low,1,Study Room,Shutdown_Study Room,Off,NaN
2,0eb984b0-2c7b-4b53-ba33-8671ea7349cb,Scene_Control,Wake up,SmartPlug_Radio,Off,Afternoon,66%,17°C,High,1,Bedroom,Wakeup_Bedroom,Off,NaN
3,570a3a08-8dfb-412b-b5b2-1f79c6c76dc4,Scene_Control,Clear bedroom state completely,SmartPlug_Air Purifier,Off,Noon,94%,16°C,Low,2,Bedroom,Shutdown_Bedroom,Off,NaN
4,3a3a10ff-9383-4e24-b063-91ee6ba297f4,Direct_Device_Control,Open the window coverings,Curtains,Close,Morning,46%,21°C,Medium,1,Bedroom,Direct_Control,Open,NaN


**Dataset dimensions**

In [3]:
print("Rows:",len(df))

print("Columns:",df.shape[1])

Rows: 743960
Columns: 14


**Dataset schema**

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 743960 entries, 0 to 743959
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Event_ID       743960 non-null  object
 1   Intent_Type    743960 non-null  object
 2   Command        743960 non-null  object
 3   Device         743960 non-null  object
 4   Current State  743960 non-null  object
 5   Time           743960 non-null  object
 6   Light          743960 non-null  object
 7   Temperature    743960 non-null  object
 8   Noise          743960 non-null  object
 9   Occupancy      743960 non-null  int64 
 10  Location       743960 non-null  object
 11  Scenario       743960 non-null  object
 12  Action         743960 non-null  object
 13  Parameters     105520 non-null  object
dtypes: int64(1), object(13)
memory usage: 79.5+ MB


**Dataset statistics**

In [5]:
summary=pd.DataFrame({

"Statistic":["Event IDs","Rows","Commands","Devices","Scenarios","Rooms","Actions"],

"Value":[

df.Event_ID.nunique(),
len(df),
df.Command.nunique(),
df.Device.nunique(),
df.Scenario.nunique(),
df.Location.nunique(),
df.Action.nunique()

]

})


summary

,Statistic,Value
0,Event IDs,200320
1,Rows,743960
2,Commands,2037
3,Devices,19
4,Scenarios,23
5,Rooms,5
6,Actions,5


**Command length analysis**

In [6]:
df["Command_Length"] = (df.Command.str.split().str.len())


df.Command_Length.describe()

count    743960.000000
mean          3.797301
std           1.180743
min           1.000000
25%           3.000000
50%           4.000000
75%           4.000000
max           9.000000
Name: Command_Length, dtype: float64

**Event reconstruction example**

In [7]:
event_id=df.Event_ID.iloc[0]

event=df[df.Event_ID==event_id]

event

,Event_ID,Intent_Type,Command,Device,Current State,Time,Light,Temperature,Noise,Occupancy,Location,Scenario,Action,Parameters,Command_Length
0,a3fdd58f-6f55-493e-ba54-5c2eb0173e82,Scene_Control,Time to get up,SmartPlug_TV,On,Morning,71%,17°C,High,2,Bedroom,Wakeup_Bedroom,On,NaN,4
167656,a3fdd58f-6f55-493e-ba54-5c2eb0173e82,Scene_Control,Time to get up,SmartPlug_Noise Machine,On,Morning,71%,17°C,High,2,Bedroom,Wakeup_Bedroom,Off,NaN,4
243232,a3fdd58f-6f55-493e-ba54-5c2eb0173e82,Scene_Control,Time to get up,AC,Off,Morning,71%,17°C,High,2,Bedroom,Wakeup_Bedroom,Off,Temp=17°C,4
334848,a3fdd58f-6f55-493e-ba54-5c2eb0173e82,Scene_Control,Time to get up,Light,Off,Morning,71%,17°C,High,2,Bedroom,Wakeup_Bedroom,On,NaN,4
386339,a3fdd58f-6f55-493e-ba54-5c2eb0173e82,Scene_Control,Time to get up,SmartPlug_Bed Lamp,Off,Morning,71%,17°C,High,2,Bedroom,Wakeup_Bedroom,Off,NaN,4
457299,a3fdd58f-6f55-493e-ba54-5c2eb0173e82,Scene_Control,Time to get up,Curtains,Open,Morning,71%,17°C,High,2,Bedroom,Wakeup_Bedroom,Open,NaN,4
596032,a3fdd58f-6f55-493e-ba54-5c2eb0173e82,Scene_Control,Time to get up,SmartPlug_Air Purifier,On,Morning,71%,17°C,High,2,Bedroom,Wakeup_Bedroom,Off,NaN,4
715073,a3fdd58f-6f55-493e-ba54-5c2eb0173e82,Scene_Control,Time to get up,SmartPlug_Radio,On,Morning,71%,17°C,High,2,Bedroom,Wakeup_Bedroom,Off,NaN,4


## VALIDATION AND QUALITY

**Missing value validation**

In [8]:
missing=df.isnull().sum()

missing

Event_ID               0
Intent_Type            0
Command                0
Device                 0
Current State          0
Time                   0
Light                  0
Temperature            0
Noise                  0
Occupancy              0
Location               0
Scenario               0
Action                 0
Parameters        638440
Command_Length         0
dtype: int64

**Missing Event ID**

In [9]:
missing_event_ids = df.Event_ID.isnull().sum()

print(
    "Missing Event IDs:",
    missing_event_ids
)

Missing Event IDs: 0


**Action validation**

In [10]:
valid_actions=["On","Off","Open","Close","Adjust"]


invalid=df[
~df.Action.isin(valid_actions)
]


len(invalid)

0

In [11]:
# Check Event ID format
import uuid


def valid_uuid(x):
    try:
        uuid.UUID(str(x))
        return True
    except:
        return False


invalid_ids = df[
~df.Event_ID.apply(valid_uuid)
]


len(invalid_ids)

0

**Context validation**

In [12]:
context_columns=[
"Time",
"Light",
"Temperature",
"Noise",
"Occupancy"
]


df[context_columns].isnull().sum()

Time           0
Light          0
Temperature    0
Noise          0
Occupancy      0
dtype: int64

**Device-action consistency**

In [13]:
invalid_light=df[
(df.Device=="Light")&(df.Action.isin(["OPEN","CLOSE"]))]

len(invalid_light)

0

**Validation report**

In [14]:
# =====================================================
# Validation Checks
# =====================================================

# Missing values (excluding Parameters)
required_columns = [
    "Event_ID",
    "Intent_Type",
    "Command",
    "Device",
    "Current State",
    "Time",
    "Light",
    "Temperature",
    "Noise",
    "Occupancy",
    "Location",
    "Scenario",
    "Action"
]

missing_required = df[required_columns].isnull().sum().sum()

missing_result = (
    "Passed"
    if missing_required == 0
    else f"Failed ({missing_required} missing values)"
)

# Event IDs
event_result = (
    "Passed (Expected multiple rows per Event_ID for multi-device events)"
)

# Invalid actions
valid_actions = {"On", "Off", "Open", "Close", "Adjust"}

invalid_actions = (~df["Action"].isin(valid_actions)).sum()

action_result = (
    "Passed"
    if invalid_actions == 0
    else f"Failed ({invalid_actions})"
)

# Context completeness
context_cols = [
    "Time",
    "Light",
    "Temperature",
    "Noise",
    "Occupancy",
    "Location"
]

context_missing = df[context_cols].isnull().sum().sum()

context_result = (
    "Passed"
    if context_missing == 0
    else f"Failed ({context_missing})"
)

# Parameters
parameter_result = (
    "Passed (Null/None permitted for non-parameterized devices)"
)

validation = pd.DataFrame({

    "Validation Check":[
        "Missing required values",
        "Event_ID structure",
        "Invalid action labels",
        "Context completeness",
        "Parameter validation"
    ],

    "Result":[
        missing_result,
        event_result,
        action_result,
        context_result,
        parameter_result
    ]

})

display(validation)

validation.to_csv(
    "../results/tables/validation_checks.csv",
    index=False
)

,Validation Check,Result
0,Missing required values,Passed
1,Event_ID structure,Passed (Expected multiple rows per Event_ID fo...
2,Invalid action labels,Passed
3,Context completeness,Passed
4,Parameter validation,Passed (Null/None permitted for non-parameteri...


## Distribution Analysis

**Time distribution**

In [15]:
time=df.Time.value_counts()


time

Time
Morning      148792
Afternoon    148792
Noon         148792
Night        148792
Evening      148792
Name: count, dtype: int64

In [16]:
# ====================================
# Event-level time distribution
# ====================================
time = (
    df.groupby("Event_ID")["Time"]
      .first()
      .value_counts()
      .reindex(["Morning", "Noon", "Afternoon", "Evening", "Night"])
)

# Save table
time_table = time.reset_index()
time_table.columns = ["Time_Period", "Count"]
time_table["Percentage"] = (
    time_table["Count"] /
    time_table["Count"].sum() * 100
).round(2)

time_table.to_csv(
    "../results/tables/time_distribution.csv",
    index=False
)

# Create figure
plt.figure(figsize=(7, 4))

time.plot(kind="bar")

plt.xlabel("Time Period")
plt.ylabel("Number of Events")
plt.title("Time Distribution")

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    "../results/figures/time_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Figure saved to: ../results/figures/time_distribution.png")
print("Table saved to: ../results/tables/time_distribution.csv")

Figure saved to: ../results/figures/time_distribution.png
Table saved to: ../results/tables/time_distribution.csv


**Scenario distribution**

In [17]:
scenario=df.Scenario.value_counts()


scenario.head(20)

Scenario
Direct_Control               127400
Shutdown_Kitchen              45760
Cooking_Kitchen               45320
Cleaning_Kitchen              41360
Shutdown_Study Room           30960
Sleep Preparation_Bedroom     30720
Reading_Bedroom               29760
Wakeup_Bedroom                29440
Shutdown_Living Room          29160
Cleaning_Bedroom              28440
Shutdown_Bedroom              28400
Cleaning_stop_Kitchen         27000
Cleaning_Living Room          25280
Study_Study Room              24640
Cleaning_stop_Living Room     24000
Cleaning_stop_Bedroom         23680
Eating_Dining Room            23520
Shutdown_Dining Room          22720
Cleaning_Study Room           22400
Cleaning_Dining Room          22120
Name: count, dtype: int64

In [18]:
# ====================================
# Scenario distribution
# ====================================
scenario = (
    df["Scenario"]
    .value_counts()
    .sort_values(ascending=False)
)

# Save table
scenario_table = scenario.reset_index()
scenario_table.columns = ["Scenario", "Count"]
scenario_table["Percentage"] = (
    scenario_table["Count"] /
    scenario_table["Count"].sum() * 100
).round(2)

scenario_table.to_csv(
    "../results/tables/scenario_distribution.csv",
    index=False
)

# Create figure
plt.figure(figsize=(10, 5))

scenario.plot(kind="bar")

plt.xlabel("Scenario")
plt.ylabel("Number of Events")
plt.title("Scenario Distribution")

plt.xticks(rotation=90)

plt.tight_layout()

plt.savefig(
    "../results/figures/scenario_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Figure saved to: ../results/figures/scenario_distribution.png")
print("Table saved to: ../results/tables/scenario_distribution.csv")

Figure saved to: ../results/figures/scenario_distribution.png
Table saved to: ../results/tables/scenario_distribution.csv


**Device distribution**

In [19]:
# ====================================
# Device distribution
# ====================================
device = (
    df["Device"]
    .value_counts()
    .sort_values(ascending=False)
)

# Save table
device_table = device.reset_index()
device_table.columns = ["Device", "Count"]
device_table["Percentage"] = (
    device_table["Count"] /
    device_table["Count"].sum() * 100
).round(2)

device_table.to_csv(
    "../results/tables/device_distribution.csv",
    index=False
)

# Create figure
plt.figure(figsize=(8, 5))

device.plot(kind="bar")

plt.xlabel("Device")
plt.ylabel("Number of Device Actions")
plt.title("Device Distribution")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig(
    "../results/figures/device_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Figure saved to: ../results/figures/device_distribution.png")
print("Table saved to: ../results/tables/device_distribution.csv")

Figure saved to: ../results/figures/device_distribution.png
Table saved to: ../results/tables/device_distribution.csv


**Action distribution**

In [20]:
action=df.Action.value_counts()


action

Action
Off       302917
On        282280
Open       59152
Adjust     55443
Close      44168
Name: count, dtype: int64

In [21]:
# ====================================
# Action distribution
# ====================================
action = (
    df["Action"]
    .value_counts()
    .sort_values(ascending=False)
)

# Save table
action_table = action.reset_index()
action_table.columns = ["Action", "Count"]
action_table["Percentage"] = (
    action_table["Count"] /
    action_table["Count"].sum() * 100
).round(2)

action_table.to_csv(
    "../results/tables/action_distribution.csv",
    index=False
)

# Create figure
plt.figure(figsize=(7,4))

action.plot(kind="bar")

plt.xlabel("Action")
plt.ylabel("Number of Device Actions")
plt.title("Action Distribution")

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    "../results/figures/action_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Figure saved to: ../results/figures/action_distribution.png")
print("Table saved to: ../results/tables/action_distribution.csv")

Figure saved to: ../results/figures/action_distribution.png
Table saved to: ../results/tables/action_distribution.csv


**Event complexity**

In [22]:
complexity=df.groupby(
"Event_ID"
).size()


complexity.describe()

count    200320.000000
mean          3.713858
std           3.725326
min           1.000000
25%           1.000000
50%           1.000000
75%           8.000000
max          13.000000
dtype: float64

In [23]:
# Event complexity (number of device actions per Event_ID)
complexity = df.groupby("Event_ID").size()

# ==============================
# Save distribution table
# ==============================
distribution = (
    complexity.value_counts()
    .sort_index()
    .reset_index()
)

distribution.columns = [
    "Actions_per_Event",
    "Number_of_Events"
]

distribution["Percentage"] = (
    distribution["Number_of_Events"]
    / distribution["Number_of_Events"].sum()
    * 100
).round(2)

distribution.to_csv(
    "../results/tables/event_complexity_distribution.csv",
    index=False
)

# ==============================
# Save summary statistics
# ==============================
summary = complexity.describe().to_frame(name="Value")
summary.loc["median"] = complexity.median()
summary.loc["mode"] = complexity.mode().iloc[0]

summary.to_csv(
    "../results/tables/event_complexity_summary.csv"
)

# ==============================
# Create and save figure
# ==============================
plt.figure(figsize=(7, 4))

complexity.hist(bins=13)

plt.xlabel("Number of Actions per Event")
plt.ylabel("Number of Events")
plt.title("Event-Level Interaction Complexity")

plt.tight_layout()

plt.savefig(
    "../results/figures/event_complexity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Figure saved to: ../results/figures/event_complexity.png")
print("Tables saved to: ../results/tables/")

Figure saved to: ../results/figures/event_complexity.png
Tables saved to: ../results/tables/
